##### This script contains the following:

#### 1. Importing libraries and datasets
#### 2. Data Wrangling and Exporting Cleaned Version
#### 3. Reshaping
#### 4. Data Split
#### 5. Creating the Keras Model
#### 6. Compiling and Running the RNN model
#### 7. Creating the Confusion Matrix
#### 8. Retesting the RNN model
#### 9. CNN Model
####      9.1 CNN Retesting

### 1. Importing libraries and datasets

In [238]:
# Importing libraries
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from numpy import unique
from numpy import reshape
from keras.models import Sequential
from keras.layers import Conv1D, Conv2D, Dense, BatchNormalization, Flatten, MaxPooling1D, Dropout, LSTM
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [239]:
# Defining a path for importing/exporting
path = r'/Users/peterguan/ClimateWins ML'

In [240]:
# Importing the ClimateWins Unscaled dataset
climatewins_unscaled = pd.read_csv(r'/Users/peterguan/ClimateWins ML/Data Sets/Dataset-weather-prediction-dataset-processed.csv', index_col = False)

In [241]:
# Checking if the dataset imported successfully 
climatewins_unscaled.head()

,DATE,MONTH,BASEL_cloud_cover,BASEL_wind_speed,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_snow_depth,BASEL_sunshine,...,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_snow_depth,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,19600101,1,7,2.1,0.85,1.018,0.32,0.09,0,0.7,...,5,0.88,1.0003,0.45,0.34,0,4.7,8.5,6.0,10.9
1,19600102,1,6,2.1,0.84,1.018,0.36,1.05,0,1.1,...,7,0.91,1.0007,0.25,0.84,0,0.7,8.9,5.6,12.1
2,19600103,1,8,2.1,0.90,1.018,0.18,0.30,0,0.0,...,7,0.91,1.0096,0.17,0.08,0,0.1,10.5,8.1,12.9
3,19600104,1,3,2.1,0.92,1.018,0.58,0.00,0,4.1,...,7,0.86,1.0184,0.13,0.98,0,0.0,7.4,7.3,10.6
4,19600105,1,6,2.1,0.95,1.018,0.65,0.14,0,5.4,...,3,0.80,1.0328,0.46,0.00,0,5.7,5.7,3.0,8.4


In [242]:
# Checking the dimensions of the dataset
climatewins_unscaled.shape

(22950, 170)

In [243]:
# Importing the PLEASANT WEATHER dataset
pleasantweather = pd.read_csv(os.path.join(path, 'Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'))

In [244]:
# Checking if the dataset imported successfully 
pleasantweather.head()

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [245]:
# Checking the dimensions of the dataset
pleasantweather.shape

(22950, 16)

### 2. Data Wrangling

In [247]:
# Dropping the "DATE" column from pleasantweather dataset

pleasantweather.drop(columns = 'DATE', inplace = True)

In [248]:
# Dropping the DATE and MONTH columns from Unscaled dataset
climatewins_unscaled = climatewins_unscaled.drop(columns=['DATE', 'MONTH'], axis=1)
climatewins_unscaled.head()

,BASEL_cloud_cover,BASEL_wind_speed,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_snow_depth,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,...,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_snow_depth,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,2.1,0.85,1.018,0.32,0.09,0,0.7,6.5,0.8,...,5,0.88,1.0003,0.45,0.34,0,4.7,8.5,6.0,10.9
1,6,2.1,0.84,1.018,0.36,1.05,0,1.1,6.1,3.3,...,7,0.91,1.0007,0.25,0.84,0,0.7,8.9,5.6,12.1
2,8,2.1,0.90,1.018,0.18,0.30,0,0.0,8.5,5.1,...,7,0.91,1.0096,0.17,0.08,0,0.1,10.5,8.1,12.9
3,3,2.1,0.92,1.018,0.58,0.00,0,4.1,6.3,3.8,...,7,0.86,1.0184,0.13,0.98,0,0.0,7.4,7.3,10.6
4,6,2.1,0.95,1.018,0.65,0.14,0,5.4,3.0,-0.7,...,3,0.80,1.0328,0.46,0.00,0,5.7,5.7,3.0,8.4


In [249]:
# Checking if the DATE column was successfully dropped
pleasantweather.head()

,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [250]:
# Checking the dimensions
pleasantweather.shape

(22950, 15)

In [251]:
# Dropping all Gdansk, Roma, Tours weather station columns from climatewins_unscaled since they are not included in pleasant weather data

cols_to_drop = [col for col in climatewins_unscaled.columns if col.startswith(('GDANSK', 'ROMA', 'TOURS'))]
climatewins_unscaled = climatewins_unscaled.drop(columns=cols_to_drop)

In [252]:
# Listing all the columns
climatewins_unscaled.columns

Index(['BASEL_cloud_cover', 'BASEL_wind_speed', 'BASEL_humidity',
       'BASEL_pressure', 'BASEL_global_radiation', 'BASEL_precipitation',
       'BASEL_snow_depth', 'BASEL_sunshine', 'BASEL_temp_mean',
       'BASEL_temp_min',
       ...
       'VALENTIA_cloud_cover', 'VALENTIA_humidity', 'VALENTIA_pressure',
       'VALENTIA_global_radiation', 'VALENTIA_precipitation',
       'VALENTIA_snow_depth', 'VALENTIA_sunshine', 'VALENTIA_temp_mean',
       'VALENTIA_temp_min', 'VALENTIA_temp_max'],
      dtype='object', length=147)

In [253]:
# Trying to find out all the different measurement types for each location

# Extract location names 
locations = set([col.split('_')[0] for col in climatewins_unscaled.columns])

# Create a dictionary to store measurement counts for each location
measurement_counts = {location: {} for location in locations}

# Count occurrences of each measurement type for each location
for col in climatewins_unscaled.columns:
    parts = col.split('_') 
    location = parts[0] 
    measurement = '_'.join(parts[1:])  # Join remaining parts if there are more than two

    if measurement not in measurement_counts[location]:
        measurement_counts[location][measurement] = 1
    else:
        measurement_counts[location][measurement] += 1

# Print the measurement counts for each location
for location, measurements in measurement_counts.items():
    print(f"Location: {location}")
    for measurement, count in measurements.items():
        print(f"  - {measurement}: {count}")
    print()

Location: VALENTIA
  - cloud_cover: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - snow_depth: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: OSLO
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - snow_depth: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: MUNCHENB
  - cloud_cover: 1
  - humidity: 1
  - global_radiation: 1
  - precipitation: 1
  - snow_depth: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: MAASTRICHT
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: MADRID
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Locatio

In [254]:
# Dropping columns for wind_speed and snow_depth measurements

# Creating a list of columns to drop
cols_to_drop = [col for col in climatewins_unscaled.columns if col.endswith(('wind_speed', 'snow_depth'))]

# Dropping
climatewins_unscaled = climatewins_unscaled.drop(columns=cols_to_drop)

In [255]:
# Checking if the columns were dropped successfully 
climatewins_unscaled.head()

,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,BASEL_temp_max,BELGRADE_cloud_cover,...,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,10.9,1,...,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,10.1,6,...,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,9.9,6,...,4.1,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,10.6,8,...,2.3,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,6.0,8,...,4.3,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


In [256]:
# There are missing measurements for Kassel's cloud cover, Stockholm's humidity, and Munchenb's pressure
# We know that Ljubljana is near Kassel, Sonnblick is near Munchenb, and Olso is close enough to Stockholm

# Define relationships between locations
location_pairs = {
    'KASSEL': 'LJUBLJANA',
    'STOCKHOLM': 'OSLO',
    'MUNCHENB': 'SONNBLICK'
}

# Define the desired order of measurements
measurement_order = ['cloud_cover', 'humidity', 'pressure', 'global_radiation', 
                     'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max']

# Function to fill missing values and insert in correct position
def fill_missing_values(climatewins_unscaled, location, measurement, neighbor):
    """
    Fills missing values for a given location and measurement using data from a neighbor location.
    Inserts the new column in the correct position based on the measurement order.

    Args:
        df_unscaled: The DataFrame containing the weather data.
        location: The location with missing values.
        measurement: The measurement with missing values.
        neighbor: The neighboring location to use for filling.

    Returns:
        The updated DataFrame with filled missing values and columns in the correct order.
    """
    source_col = f'{neighbor}_{measurement}'
    target_col = f'{location}_{measurement}'

    # Determine the insertion index 
    if measurement == measurement_order[0]:  # If it's the first measurement for the location
        # Find the index of the first column for the location (or 0 if no location columns exist)
        location_columns = [col for col in climatewins_unscaled.columns if col.startswith(location)]
        if location_columns:
            insert_index = climatewins_unscaled.columns.get_loc(location_columns[0]) 
        else:
            insert_index = 0
    else:
        insert_index = climatewins_unscaled.columns.get_loc(f'{location}_{measurement_order[measurement_order.index(measurement) - 1]}') + 1 

    # Create the new column with missing values and insert it at the correct position
    climatewins_unscaled.insert(insert_index, target_col, np.nan) 

    # Fill missing values in the new column
    climatewins_unscaled[target_col].fillna(climatewins_unscaled[source_col], inplace=True) 

    return climatewins_unscaled

# Fill missing values for each location and measurement
for location, neighbor in location_pairs.items():
    for measurement in measurement_order:
        if f'{location}_{measurement}' not in climatewins_unscaled.columns:  # Check if column already exists
            climatewins_unscaled = fill_missing_values(climatewins_unscaled, location, measurement, neighbor)

# Checking new columns for existance and location
selected_columns = [col for col in climatewins_unscaled.columns if col.startswith(('KASSEL', 'STOCKHOLM', 'MUNCHENB'))]
print(climatewins_unscaled[selected_columns])

       KASSEL_cloud_cover  KASSEL_humidity  KASSEL_pressure  \
0                     8.0             0.82           1.0094   
1                     6.0             0.86           1.0086   
2                     8.0             0.91           1.0129   
3                     6.0             0.87           1.0290   
4                     7.0             0.86           1.0262   
...                   ...              ...              ...   
22945                 4.0             0.77           1.0161   
22946                 3.0             0.77           1.0161   
22947                 3.0             0.77           1.0161   
22948                 3.0             0.77           1.0161   
22949                 3.0             0.77           1.0161   

       KASSEL_global_radiation  KASSEL_precipitation  KASSEL_sunshine  \
0                         0.28                  0.48              1.6   
1                         0.12                  0.27              0.0   
2                       

/var/folders/j8/l8hsy2p15dj9phf2ztxh0b040000gn/T/ipykernel_88519/2400607794.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  climatewins_unscaled[target_col].fillna(climatewins_unscaled[source_col], inplace=True)
/var/folders/j8/l8hsy2p15dj9phf2ztxh0b040000gn/T/ipykernel_88519/2400607794.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate o

In [257]:
# Checking new columns for existance and location
selected_columns = [col for col in climatewins_unscaled.columns if col.startswith(('MUNCHENB'))]
print(climatewins_unscaled[selected_columns])

       MUNCHENB_cloud_cover  MUNCHENB_humidity  MUNCHENB_pressure  \
0                         5               0.67             1.0304   
1                         6               0.72             1.0292   
2                         6               0.91             1.0320   
3                         6               0.90             1.0443   
4                         5               0.85             1.0430   
...                     ...                ...                ...   
22945                     2               0.76             1.0263   
22946                     6               0.70             1.0263   
22947                     7               0.64             1.0263   
22948                     6               0.75             1.0263   
22949                     5               0.83             1.0263   

       MUNCHENB_global_radiation  MUNCHENB_precipitation  MUNCHENB_sunshine  \
0                           0.20                    0.10                0.0   
1            

In [258]:
# Checking the dimensions of the climatewins_unscaled dataset
climatewins_unscaled.shape

(22950, 135)

In [259]:
# Checking the dimensions of the pleasantweather dataset
pleasantweather.shape

(22950, 15)

In [260]:
# Export cleaned weather data
climatewins_unscaled.to_csv(os.path.join(path, 'climatewins_unscaled_clean.csv'), index=False)

### 3. Reshaping

In [262]:
X = climatewins_unscaled

In [263]:
# Assigning the pleasantweather dataset to 'y'
y = pleasantweather

In [264]:
# Turning X and y to arrays

X = np.array(X)
y = np.array(y)

In [265]:
# Reshaping X

X = X.reshape(-1,15,9)

In [266]:
# Checking the RESHAPE

X

array([[[  7.    ,   0.85  ,   1.018 , ...,   6.5   ,   0.8   ,
          10.9   ],
        [  1.    ,   0.81  ,   1.0195, ...,   3.7   ,  -0.9   ,
           7.9   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.4   ,  -0.4   ,
           5.1   ],
        ...,
        [  4.    ,   0.73  ,   1.0304, ...,  -5.9   ,  -8.5   ,
          -3.2   ],
        [  5.    ,   0.98  ,   1.0114, ...,   4.2   ,   2.2   ,
           4.9   ],
        [  5.    ,   0.88  ,   1.0003, ...,   8.5   ,   6.    ,
          10.9   ]],

       [[  6.    ,   0.84  ,   1.018 , ...,   6.1   ,   3.3   ,
          10.1   ],
        [  6.    ,   0.84  ,   1.0172, ...,   2.9   ,   2.2   ,
           4.4   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.3   ,   1.4   ,
           3.1   ],
        ...,
        [  6.    ,   0.97  ,   1.0292, ...,  -9.5   , -10.5   ,
          -8.5   ],
        [  5.    ,   0.62  ,   1.0114, ...,   4.    ,   3.    ,
           5.    ],
        [  7.    ,   0.91  ,   1.0007, ...,   8.

In [267]:
X.shape

(22950, 15, 9)

In [268]:
y.shape

(22950, 15)

### 4. Data Split

In [270]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [271]:
# Checking the dimensions
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212, 15)
(5738, 15, 9) (5738, 15)


### 5. Creating the Keras Model

In [273]:
epochs = 15
batch_size = 8
n_hidden = 8

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [274]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 8)              │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 15)             │           135 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 711 (2.78 KB)

 Trainable params: 711 (2.78 KB)

 Non-trainable params: 0 (0.00 B)

### 6. Compiling and Running the RNN Model

In [276]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [277]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/15
2152/2152 - 2s - 1ms/step - accuracy: 0.1028 - loss: 9.1829
Epoch 2/15
2152/2152 - 2s - 926us/step - accuracy: 0.2098 - loss: 9.3388
Epoch 3/15
2152/2152 - 2s - 935us/step - accuracy: 0.2757 - loss: 9.6827
Epoch 4/15
2152/2152 - 2s - 910us/step - accuracy: 0.2809 - loss: 10.1171
Epoch 5/15
2152/2152 - 2s - 904us/step - accuracy: 0.2914 - loss: 10.4805
Epoch 6/15
2152/2152 - 2s - 920us/step - accuracy: 0.3092 - loss: 10.9096
Epoch 7/15
2152/2152 - 2s - 953us/step - accuracy: 0.3514 - loss: 11.3332
Epoch 8/15
2152/2152 - 2s - 910us/step - accuracy: 0.4070 - loss: 11.7383
Epoch 9/15
2152/2152 - 2s - 907us/step - accuracy: 0.4454 - loss: 12.0118
Epoch 10/15
2152/2152 - 2s - 904us/step - accuracy: 0.4886 - loss: 12.4081
Epoch 11/15
2152/2152 - 2s - 918us/step - accuracy: 0.5151 - loss: 12.7456
Epoch 12/15
2152/2152 - 2s - 918us/step - accuracy: 0.5300 - loss: 13.0371
Epoch 13/15
2152/2152 - 2s - 913us/step - accuracy: 0.5523 - loss: 13.4585
Epoch 14/15
2152/2152 - 2s - 914us/step

### 7. Creating the Confusion Matrix

In [279]:
# Define list of stations names

stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'

}

In [280]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [281]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step
Pred        BASEL  MADRID
True                     
BASEL        3677       5
BELGRADE     1084       8
BUDAPEST      212       2
DEBILT         82       0
DUSSELDORF     29       0
HEATHROW       82       0
KASSEL         11       0
LJUBLJANA      59       2
MAASTRICHT      9       0
MADRID        450       8
MUNCHENB        8       0
OSLO            5       0
STOCKHOLM       4       0
VALENTIA        0       1


### 8. RNN Retesting

In [283]:
epochs = 30
batch_size = 16
n_hidden = 32

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [284]:
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 32)             │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,871 (22.93 KB)

 Trainable params: 5,871 (22.93 KB)

 Non-trainable params: 0 (0.00 B)

In [285]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [286]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 2s - 1ms/step - accuracy: 0.1836 - loss: 10.4834
Epoch 2/30
1076/1076 - 1s - 1ms/step - accuracy: 0.2862 - loss: 10.5347
Epoch 3/30
1076/1076 - 1s - 1ms/step - accuracy: 0.3094 - loss: 10.6873
Epoch 4/30
1076/1076 - 1s - 1ms/step - accuracy: 0.4958 - loss: 10.9264
Epoch 5/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6086 - loss: 11.1394
Epoch 6/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6353 - loss: 11.4569
Epoch 7/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6410 - loss: 11.6613
Epoch 8/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6430 - loss: 12.0095
Epoch 9/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6432 - loss: 12.3166
Epoch 10/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6430 - loss: 12.6188
Epoch 11/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6430 - loss: 13.0651
Epoch 12/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6439 - loss: 13.3476
Epoch 13/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6439 - loss: 13.6408
Epoch 14/30
1076/1076 - 1s - 1ms/step - accuracy: 0.6440 - l

In [287]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [288]:
# Evaluate

print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step
Pred        BASEL  MADRID
True                     
BASEL        3681       1
BELGRADE     1092       0
BUDAPEST      214       0
DEBILT         82       0
DUSSELDORF     29       0
HEATHROW       82       0
KASSEL         11       0
LJUBLJANA      61       0
MAASTRICHT      9       0
MADRID        458       0
MUNCHENB        8       0
OSLO            5       0
STOCKHOLM       4       0
VALENTIA        1       0


### 9. CNN Model

In [290]:
epochs = 30
batch_size = 16
n_hidden = 32

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax'))

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [291]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 14, 32)         │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 14, 16)         │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,831 (11.06 KB)

 Trainable params: 2,831 (11.06 KB)

 Non-trainable params: 0 (0.00 B)

In [292]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [293]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/30
1076/1076 - 1s - 562us/step - accuracy: 0.1224 - loss: 4676.5996
Epoch 2/30
1076/1076 - 0s - 330us/step - accuracy: 0.1398 - loss: 50894.0430
Epoch 3/30
1076/1076 - 0s - 329us/step - accuracy: 0.1378 - loss: 176993.7188
Epoch 4/30
1076/1076 - 0s - 331us/step - accuracy: 0.1386 - loss: 397370.1250
Epoch 5/30
1076/1076 - 0s - 332us/step - accuracy: 0.1323 - loss: 737113.6875
Epoch 6/30
1076/1076 - 0s - 331us/step - accuracy: 0.1328 - loss: 1156242.3750
Epoch 7/30
1076/1076 - 0s - 331us/step - accuracy: 0.1312 - loss: 1688283.0000
Epoch 8/30
1076/1076 - 0s - 331us/step - accuracy: 0.1353 - loss: 2369655.2500
Epoch 9/30
1076/1076 - 0s - 331us/step - accuracy: 0.1343 - loss: 3180668.5000
Epoch 10/30
1076/1076 - 0s - 331us/step - accuracy: 0.1289 - loss: 4084749.5000
Epoch 11/30
1076/1076 - 0s - 330us/step - accuracy: 0.1293 - loss: 5193961.0000
Epoch 12/30
1076/1076 - 0s - 344us/step - accuracy: 0.1301 - loss: 6517350.5000
Epoch 13/30
1076/1076 - 0s - 339us/step - accuracy: 0.132

In [294]:
# Creating the confusion matrix
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [295]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 307us/step
Pred        BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                   
BASEL           1044         1      31         844       440     230   
BELGRADE         566         0       2         274       137       6   
BUDAPEST          91         1       1          50        48       0   
DEBILT            25         1       1          21        26       0   
DUSSELDORF         6         0       0          10         9       0   
HEATHROW          14         0       1           9        39       3   
KASSEL             4         0       0           3         2       0   
LJUBLJANA         28         0       0           3        10       0   
MAASTRICHT         4         0       0           2         1       0   
MADRID            99         0       5          55       128      14   
MUNCHENB           6         0       0           0         0       0   
OSLO               1 

### 9.1 CNN Retesting

In [297]:
epochs = 15
batch_size = 8
n_hidden = 8

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='tanh'))

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [298]:
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_5 (Conv1D)               │ (None, 14, 8)          │           152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 14, 16)         │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,991 (7.78 KB)

 Trainable params: 1,991 (7.78 KB)

 Non-trainable params: 0 (0.00 B)

In [299]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [300]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/15
2152/2152 - 1s - 417us/step - accuracy: 0.1919 - loss: 27.9759
Epoch 2/15
2152/2152 - 1s - 299us/step - accuracy: 0.1945 - loss: 29.9054
Epoch 3/15
2152/2152 - 1s - 300us/step - accuracy: 0.1945 - loss: 29.9063
Epoch 4/15
2152/2152 - 1s - 299us/step - accuracy: 0.1945 - loss: 29.9184
Epoch 5/15
2152/2152 - 1s - 299us/step - accuracy: 0.1945 - loss: 29.9072
Epoch 6/15
2152/2152 - 1s - 300us/step - accuracy: 0.1946 - loss: 29.9035
Epoch 7/15
2152/2152 - 1s - 299us/step - accuracy: 0.1946 - loss: 29.9325
Epoch 8/15
2152/2152 - 1s - 299us/step - accuracy: 0.1946 - loss: 29.9372
Epoch 9/15
2152/2152 - 1s - 299us/step - accuracy: 0.1946 - loss: 29.9381
Epoch 10/15
2152/2152 - 1s - 301us/step - accuracy: 0.1946 - loss: 29.9390
Epoch 11/15
2152/2152 - 1s - 300us/step - accuracy: 0.1959 - loss: 29.7302
Epoch 12/15
2152/2152 - 1s - 300us/step - accuracy: 0.4891 - loss: 26.9649
Epoch 13/15
2152/2152 - 1s - 300us/step - accuracy: 0.6001 - loss: 31.2950
Epoch 14/15
2152/2152 - 1s - 303us

In [301]:
# Creating the confusion matrix
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

In [302]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 272us/step
Pred        BASEL  BELGRADE
True                       
BASEL        3504       178
BELGRADE     1085         7
BUDAPEST      214         0
DEBILT         82         0
DUSSELDORF     29         0
HEATHROW       82         0
KASSEL         11         0
LJUBLJANA      61         0
MAASTRICHT      9         0
MADRID        458         0
MUNCHENB        8         0
OSLO            5         0
STOCKHOLM       4         0
VALENTIA        1         0
